In [ ]:
def lasso(Y, X, beta, l = 0.01):
    N = len(Y)
    Y_pred = pred(X, beta)
    lasso = (RSS(Y, Y_pred))/(2*N) + l*np.sum(abs(beta[1:]))
    return lasso

def grad(Y, X, beta, l):
    N = len(Y)
    XtY = np.matmul(np.transpose(X), Y)
    XtX = np.matmul(np.transpose(X), X)
    #Removing beta0 from the regularisation
    betanew = list(np.sign(beta[1:]))
    betanew.insert(0,0)
    d = -1*(XtY - np.dot(XtX, beta))/N + l*np.array(betanew)
    return d

#A function to add into the stopping criteria 
def lasso_diff(Y, X, beta, step, d, l):
    return abs((lasso(Y, X, beta, l) - lasso(Y, X, beta + step*d, l))/lasso(Y, X, beta, l))

def grad_descent(Y, X, beta, l):
    step = 1
    d = -1 * grad(Y, X, beta, l)
    counter = 1
    #10,000 iterations unless the difference in loss is minimal
    while (counter < 1000) and lasso_diff(Y, X, beta, step, d, l) > 1e-8:
        beta = beta + step*d
        #Descent diretion
        d = -1 * grad(Y, X, beta, l)
        counter += 1
        #Step size proportional to the number of iterations
        step = 10/counter
    return beta

#Cross Validation 
def cross_val(Y, X, folds, train_func, performance, l, *args, k=5):
    score = 0
    for i in range(len(folds)):
        #One of the folds picked as a validation set
        val_indexes = folds[i]
        #The rest of the folds are left as training sets
        train_indexes = list(set(range(Y.shape[0])) - set(val_indexes))
        
        X_train_i = X[train_indexes, :]
        y_train_i = Y[train_indexes]

        X_val_i = X[val_indexes, :] 
        y_val_i = Y[val_indexes] 

        beta = train_func(y_train_i, X_train_i, np.ones(7), l, *args)
        Y_pred = pred(X_val_i, beta)
        perform_val = performance(y_val_i, Y_pred) 
        score += perform_val
    # Return the average score
    return score/len(folds)



In [ ]:
#Indexes for the folds 
folds_indexes = np.split(np.random.permutation(np.arange(len(y))), 5)

def elastic(Y, X,  beta, l=0.01, alpha = 0.01):
    #returns value of elastic nets function
    Y_pred = pred(X, beta)
    return RSS(Y, Y_pred)/2*N + l*(alpha*np.sum(beta[1:]) + (1-alpha)*np.sqrt(np.dot(beta[1:], beta[1:])))

#Stopping function similar to that in lasso
def nets_diff(Y, X, beta, step, d, l, alpha):
    return abs(elastic(Y, X, beta, l, alpha) - elastic(Y, X, beta + step*d, l, alpha))

def grad2(Y, X, beta, l, alpha = 0.1):
    #Gradient of elastic nets
    N = len(Y)
    XtY = np.matmul(np.transpose(X), Y)
    XtX = np.matmul(np.transpose(X), X)
    #Beta0 term not included in the shrinkage
    betanew = list((beta[1:]))
    betanew.insert(0, 0)
    betanew = np.array(betanew)
    d = -1*(XtY - np.dot(XtX, beta))/N + l*(alpha * np.sign(betanew) + 2*(1-alpha)*(betanew))
    return d

def grad_descent2(Y, X, beta, l, alpha):
    #Gradient descent applied to elastic nets loss function
    step = 1
    d = -1 * grad2(Y, X, beta, l)
    counter = 1
    #10,000 iterations unless the difference in loss is minimal
    while (counter < 1000) and nets_diff(Y, X, beta, step, d, l, alpha) > 1e-8:
        beta = beta + step*d
        d = -1 * grad2(Y, X, beta, l, alpha)
        counter += 1
        step = 10/counter
    return beta

#grad_descent2(Y, Xst, np.ones(7), l=0.1, alpha = 0.5)
def grid_search_EN(y, X, folds, train_func, performance, lambda_vals, alpha_vals, k=5):
    best_vals = []
    best_val = 10
    #Loop over values of lambda for each value of alpha
    for a in alpha_vals:
        for l in lambda_vals:
            #Score for a given (l, a)
            perform_val = cross_val(y, X, folds, train_func, performance, l, a, k=5)
            #update to find smallest MSE
            if perform_val < best_val:
                best_val = perform_val
                best_l = l
        #Save the score and the best lambda for each alpha
        best_vals.append([best_val, a, best_l])
        #Reset the score to something high
        best_val = 10
    return best_vals


alpha_vals = np.array((0.1, 0.5, 0.9))
EN_lambda_vals = np.linspace(0, 0.01, 201)
optimal_values_EN = grid_search_EN(y, X, folds_indexes, grad_descent2, MSE, EN_lambda_vals, alpha_vals)
